# BTCUSDC Order-Flow Study: Sudden Burst Diagnostics

This notebook is for visual inspection of sudden bursts in market orders.

The first question is deliberately visual:

> what happens when the tape is calm, then a compressed burst of buy or sell market orders arrives?

For a last-`N` trade window, the notebook plots:

- midprice,
- event-time trade-flow imbalance,
- trade arrival intensity in trades/sec.

The default signal is sign-only, `a = 0`, because this notebook is about bursts of order direction before adding more signal complexity.

## Definitions

For each trade `t` and event window `N`:

```text
net_count_N(t) = sum(sign over last N trades)
signed_share_N(t) = net_count_N(t) / N
span_sec_N(t) = timestamp(t) - timestamp(t - N + 1)
trades_per_sec_N(t) = N / span_sec_N(t)
trades_per_sec_baseline_N(t) = median of prior trades_per_sec_N values
trades_per_sec_to_baseline_N(t) = trades_per_sec_N(t) / trades_per_sec_baseline_N(t)
volume_imbalance_N(t) = signed_volume_N(t) / total_volume_N(t)
volume_to_baseline_N(t) = total_volume_N(t) / volume_baseline_N(t)
```

A sudden burst candidate is marked when:

```text
trades_per_sec_N >= high activity threshold
previous calm intensity <= calm threshold
abs(signed_share_N) >= imbalance threshold
```

Because rolling event windows overlap, the notebook also applies a cooldown and marks only burst starts on the price chart.

Plot controls:

- `N trades`: last-`N` window used for imbalance, volume, and intensity.
- `imb th`: minimum absolute signed share required for a directional burst.
- `rate q`: quantile used to define high activity in trades/sec.
- `calm q`: quantile used to define the prior calm regime.
- `calm min`: lookback length for the calm-state median in minutes.
- `cooldown s`: suppression window after a burst start so the same cluster is not marked repeatedly.

The thresholds are intentionally simple so the plot is easy to reason about. They are not a trading rule.

In [1]:
from pathlib import Path
import json
import sys
import uuid

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import HTML, display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [2]:
def find_backtester_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "stats").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the exchange-data-backtester project root")


PROJECT_ROOT = find_backtester_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stats.notebook import load_orderflow_day


In [3]:
EXCHANGE = "binance"
SYMBOL = "BTCUSDC"
REFERENCE_DAY = "20260223"
REPLAY_ON_GAP = "skip-segment"

# Interactive plotting defaults.
DEFAULT_EVENT_N = 100
DEFAULT_IMBALANCE_THRESHOLD = 0.60
DEFAULT_INTENSITY_TAIL_Q = 0.90
DEFAULT_CALM_Q = 0.50
DEFAULT_CALM_LOOKBACK_MIN = 5
DEFAULT_BURST_COOLDOWN_SEC = 60
DEFAULT_VOLUME_BASELINE_WINDOWS = 20
MAX_PLOT_ROWS = 60_000

_, trades, top, day_summary = load_orderflow_day(
    day=REFERENCE_DAY,
    symbol=SYMBOL,
    exchange=EXCHANGE,
    replay_on_gap=REPLAY_ON_GAP,
)

display(day_summary.to_frame("value"))


,value
exchange,binance
symbol,BTCUSDC
day,20260223
day_dir,/Users/hoangdeveloper/PycharmProjects/exchange...
trades_rows,868008
top_rows,722893
trade_start_utc,2026-02-23 01:00:04.958000+00:00
trade_end_utc,2026-02-23 23:15:00.320000+00:00
top_start_utc,2026-02-23 01:00:04.735000+00:00
top_end_utc,2026-02-23 23:14:59.835000+00:00


## Build Trade-Aligned Price Data

Each trade gets the latest known top-of-book midprice at or before the trade timestamp. The price panel therefore uses midprice, not trade price.

In [4]:
from stats.features import make_trade_frame

trade_frame = make_trade_frame(trades, top)
summary = pd.Series({
    "trade_frame_rows": len(trade_frame),
    "start_utc": trade_frame["ts"].min(),
    "end_utc": trade_frame["ts"].max(),
    "buy_trades": int((trade_frame["aggr_sign"] > 0).sum()),
    "sell_trades": int((trade_frame["aggr_sign"] < 0).sum()),
})
display(summary.to_frame("value"))
display(trade_frame.head())


,value
trade_frame_rows,868008
start_utc,2026-02-23 01:00:04.958000+00:00
end_utc,2026-02-23 23:15:00.320000+00:00
buy_trades,445726
sell_trades,422282


,ts,price,qty,aggr_sign,mid_at_book
0,2026-02-23 01:00:04.958000+00:00,66704.44,0.00009,1.0,66704.435
1,2026-02-23 01:00:04.958000+00:00,66704.44,0.00017,1.0,66704.435
2,2026-02-23 01:00:05.037000+00:00,66704.44,0.00989,1.0,66704.435
3,2026-02-23 01:00:05.087000+00:00,66704.43,0.00123,-1.0,66704.435
4,2026-02-23 01:00:05.087000+00:00,66702.01,0.00008,-1.0,66704.435


## Event-Time Burst Features

This section computes last-`N` trade imbalance and the clock-time compression of those `N` trades. High trades/sec means the same number of events arrived in a shorter wall-clock interval.

In [5]:
def _sample_evenly(frame: pd.DataFrame, *, max_rows: int) -> pd.DataFrame:
    if len(frame) <= max_rows:
        return frame.copy()
    step = int(np.ceil(len(frame) / max_rows))
    return frame.iloc[::step].copy()


def build_event_burst_features(trade_frame: pd.DataFrame, *, event_n: int) -> pd.DataFrame:
    event_n = int(event_n)
    if event_n <= 1:
        raise ValueError("event_n must be greater than 1")
    if len(trade_frame) < event_n:
        raise ValueError(f"not enough trades for event_n={event_n}")

    out = trade_frame[["ts", "mid_at_book", "qty", "aggr_sign"]].copy().sort_values("ts").reset_index(drop=True)
    signs = out["aggr_sign"].to_numpy(dtype=float)
    qty = out["qty"].to_numpy(dtype=float)

    out["net_count_N"] = pd.Series(signs).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["signed_share_N"] = out["net_count_N"] / float(event_n)
    out["total_volume_N"] = pd.Series(qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["signed_volume_N"] = pd.Series(signs * qty).rolling(event_n, min_periods=event_n).sum().to_numpy()
    out["volume_imbalance_N"] = out["signed_volume_N"] / out["total_volume_N"]
    out["volume_baseline_N"] = (
        pd.Series(out["total_volume_N"])
        .rolling(DEFAULT_VOLUME_BASELINE_WINDOWS, min_periods=max(5, DEFAULT_VOLUME_BASELINE_WINDOWS // 2))
        .median()
        .shift(1)
        .to_numpy()
    )
    out["volume_to_baseline_N"] = out["total_volume_N"] / out["volume_baseline_N"]

    ts_ns = pd.to_datetime(out["ts"], utc=True).astype("int64").to_numpy()
    span_ns = np.full(len(out), np.nan, dtype=float)
    span_ns[event_n - 1 :] = ts_ns[event_n - 1 :] - ts_ns[: len(out) - event_n + 1]
    span_sec = span_ns / 1e9
    span_sec[span_sec <= 0] = np.nan
    out["event_span_sec_N"] = span_sec
    out["trades_per_sec_N"] = float(event_n) / out["event_span_sec_N"]
    out["trades_per_sec_baseline_N"] = (
        pd.Series(out["trades_per_sec_N"])
        .rolling(DEFAULT_VOLUME_BASELINE_WINDOWS, min_periods=max(5, DEFAULT_VOLUME_BASELINE_WINDOWS // 2))
        .median()
        .shift(1)
        .to_numpy()
    )
    out["trades_per_sec_to_baseline_N"] = out["trades_per_sec_N"] / out["trades_per_sec_baseline_N"]
    out["signed_flow_rate_N"] = out["net_count_N"] / out["event_span_sec_N"]
    return out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


def add_sudden_burst_flags(
    features: pd.DataFrame,
    *,
    imbalance_threshold: float,
    intensity_tail_q: float,
    calm_q: float,
    calm_lookback_min: int,
    cooldown_sec: int,
) -> tuple[pd.DataFrame, dict]:
    if not 0 <= imbalance_threshold <= 1:
        raise ValueError("imbalance_threshold must be between 0 and 1")
    if not 0 < intensity_tail_q < 1:
        raise ValueError("intensity_tail_q must be between 0 and 1")
    if not 0 < calm_q < 1:
        raise ValueError("calm_q must be between 0 and 1")

    out = features.copy().sort_values("ts")
    intensity = out["trades_per_sec_N"].astype(float)
    high_cutoff = float(intensity.quantile(intensity_tail_q))
    calm_cutoff = float(intensity.quantile(calm_q))

    prior_calm = (
        out.set_index("ts")["trades_per_sec_N"]
        .rolling(f"{int(calm_lookback_min)}min", min_periods=5)
        .median()
        .shift(1)
        .reset_index(drop=True)
    )
    out["prior_calm_trades_per_sec_N"] = prior_calm.to_numpy(dtype=float)
    out["is_high_intensity"] = out["trades_per_sec_N"] >= high_cutoff
    out["was_calm"] = out["prior_calm_trades_per_sec_N"] <= calm_cutoff
    out["is_directional"] = out["signed_share_N"].abs() >= float(imbalance_threshold)
    out["is_sudden_burst"] = out["is_high_intensity"] & out["was_calm"] & out["is_directional"]
    out["burst_side"] = np.where(out["signed_share_N"] > 0, "buy", "sell")

    cooldown_ns = int(pd.Timedelta(seconds=int(cooldown_sec)).value)
    ts_ns = pd.to_datetime(out["ts"], utc=True).astype("int64").to_numpy()
    burst_start = np.zeros(len(out), dtype=bool)
    next_allowed_ts = np.iinfo(np.int64).min
    for idx, (is_burst, ts_value) in enumerate(zip(out["is_sudden_burst"].to_numpy(dtype=bool), ts_ns)):
        if is_burst and ts_value >= next_allowed_ts:
            burst_start[idx] = True
            next_allowed_ts = ts_value + cooldown_ns
    out["is_sudden_burst_start"] = burst_start

    thresholds = {
        "high_intensity_cutoff": high_cutoff,
        "calm_intensity_cutoff": calm_cutoff,
        "imbalance_threshold": float(imbalance_threshold),
        "cooldown_sec": int(cooldown_sec),
        "raw_sudden_burst_windows": int(out["is_sudden_burst"].sum()),
        "sudden_burst_start_count": int(out["is_sudden_burst_start"].sum()),
        "buy_burst_start_count": int((out["is_sudden_burst_start"] & (out["signed_share_N"] > 0)).sum()),
        "sell_burst_start_count": int((out["is_sudden_burst_start"] & (out["signed_share_N"] < 0)).sum()),
    }
    return out, thresholds


def make_sudden_burst_figure(
    features: pd.DataFrame,
    *,
    thresholds: dict,
    title: str,
    max_plot_rows: int = MAX_PLOT_ROWS,
) -> go.Figure:
    plot_df = _sample_evenly(features, max_rows=max_plot_rows)
    burst_df = features.loc[features["is_sudden_burst_start"]].copy()
    buy_bursts = burst_df.loc[burst_df["signed_share_N"] > 0]
    sell_bursts = burst_df.loc[burst_df["signed_share_N"] < 0]

    fig = make_subplots(
        rows=7,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.018,
        row_heights=[0.30, 0.095, 0.095, 0.105, 0.105, 0.14, 0.16],
        subplot_titles=(
            "midprice",
            "last-N signed share",
            "last-N trades/sec",
            "last-N trades/sec to baseline",
            "last-N total volume",
            "last-N volume imbalance",
            "volume to baseline",
        ),
    )

    fig.add_trace(
        go.Scattergl(
            x=plot_df["ts"],
            y=plot_df["mid_at_book"],
            mode="lines",
            line=dict(color="#111827", width=1),
            name="midprice",
        ),
        row=1,
        col=1,
    )
    for data, name, color in [
        (buy_bursts, "buy sudden burst", "#16a34a"),
        (sell_bursts, "sell sudden burst", "#dc2626"),
    ]:
        fig.add_trace(
            go.Scattergl(
                x=data["ts"],
                y=data["mid_at_book"],
                mode="markers",
                marker=dict(color=color, size=7, opacity=0.85),
                name=name,
                customdata=np.stack([
                    data["signed_share_N"],
                    data["trades_per_sec_N"],
                    data["event_span_sec_N"],
                ], axis=-1) if len(data) else None,
                hovertemplate=(
                    "%{x}<br>mid=%{y:.2f}"
                    "<br>signed_share=%{customdata[0]:.3f}"
                    "<br>trades/sec=%{customdata[1]:.2f}"
                    "<br>span_sec=%{customdata[2]:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=1,
        )

    fig.add_trace(
        go.Scattergl(
            x=plot_df["ts"],
            y=plot_df["signed_share_N"],
            mode="lines",
            line=dict(color="#2563eb", width=1),
            name="signed_share_N",
        ),
        row=2,
        col=1,
    )
    imb = thresholds["imbalance_threshold"]
    for y, color in [(imb, "#16a34a"), (-imb, "#dc2626"), (0.0, "#6b7280")]:
        fig.add_hline(y=y, line_width=1, line_dash="dash", line_color=color, row=2, col=1)

    fig.add_trace(
        go.Scattergl(
            x=plot_df["ts"],
            y=plot_df["trades_per_sec_N"],
            mode="lines",
            line=dict(color="#ca8a04", width=1),
            name="trades_per_sec_N",
        ),
        row=3,
        col=1,
    )
    fig.add_hline(
        y=thresholds["high_intensity_cutoff"],
        line_width=1,
        line_dash="dash",
        line_color="#ca8a04",
        row=3,
        col=1,
    )
    fig.add_hline(
        y=thresholds["calm_intensity_cutoff"],
        line_width=1,
        line_dash="dot",
        line_color="#64748b",
        row=3,
        col=1,
    )

    fig.add_trace(
        go.Scattergl(
            x=plot_df["ts"],
            y=plot_df["trades_per_sec_to_baseline_N"],
            mode="lines",
            line=dict(color="#d97706", width=1),
            name="trades_per_sec_to_baseline_N",
        ),
        row=4,
        col=1,
    )
    fig.add_hline(y=1.0, line_width=1, line_dash="dash", line_color="#d97706", row=4, col=1)

    fig.add_trace(
        go.Scattergl(
            x=plot_df["ts"],
            y=plot_df["total_volume_N"],
            mode="lines",
            line=dict(color="#0f766e", width=1),
            name="total_volume_N",
        ),
        row=5,
        col=1,
    )

    fig.add_trace(
        go.Scattergl(
            x=plot_df["ts"],
            y=plot_df["volume_imbalance_N"],
            mode="lines",
            line=dict(color="#ef4444", width=1),
            name="volume_imbalance_N",
        ),
        row=6,
        col=1,
    )
    fig.add_hline(y=0.0, line_width=1, line_dash="dash", line_color="#6b7280", row=6, col=1)

    fig.add_trace(
        go.Scattergl(
            x=plot_df["ts"],
            y=plot_df["volume_to_baseline_N"],
            mode="lines",
            line=dict(color="#7c3aed", width=1),
            name="volume_to_baseline_N",
        ),
        row=7,
        col=1,
    )
    fig.add_hline(y=1.0, line_width=1, line_dash="dash", line_color="#7c3aed", row=7, col=1)

    hover_x = plot_df["ts"].iloc[0]
    for row in range(1, 8):
        yref = "y domain" if row == 1 else f"y{row} domain"
        fig.add_shape(
            type="line",
            xref="x",
            yref=yref,
            x0=hover_x,
            x1=hover_x,
            y0=0,
            y1=1,
            line=dict(color="#6b7280", width=1, dash="dash"),
            visible=False,
            layer="above",
        )

    fig.update_layout(
        title=title,
        height=1500,
        hovermode="x",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=60, r=30, t=90, b=50),
    )
    fig.update_yaxes(title_text="mid", row=1, col=1)
    fig.update_yaxes(title_text="signed share", row=2, col=1)
    fig.update_yaxes(title_text="trades/sec", row=3, col=1)
    fig.update_yaxes(title_text="trades/sec / baseline", row=4, col=1)
    fig.update_yaxes(title_text="volume", row=5, col=1)
    fig.update_yaxes(title_text="volume imbalance", row=6, col=1)
    fig.update_yaxes(title_text="volume / baseline", row=7, col=1)
    fig.update_xaxes(title_text="UTC time", row=7, col=1)
    return fig


def render_shared_hover_figure(fig: go.Figure) -> None:
    wrapper_id = f"burst-hover-{uuid.uuid4().hex}"
    graph_html = fig.to_html(
        full_html=False,
        include_plotlyjs=True,
        config={"responsive": True, "scrollZoom": True, "displaylogo": False},
    )
    html = f"""
<div id="{wrapper_id}" style="margin-bottom: 1rem;">
  {graph_html}
</div>
<script>
(function() {{
  const root = document.getElementById({json.dumps(wrapper_id)});
  if (!root) {{
    return;
  }}

  function attach() {{
    const graph = root.querySelector('.plotly-graph-div');
    if (!graph || typeof graph.on !== 'function') {{
      setTimeout(attach, 50);
      return;
    }}

    const guideCount = 7;
    const shapeStart = Array.isArray(graph.layout.shapes) ? graph.layout.shapes.length - guideCount : -1;
    if (shapeStart < 0) {{
      return;
    }}

    const shapeIndices = Array.from({{ length: guideCount }}, (_, idx) => shapeStart + idx);

    function setLine(xValue, visible) {{
      const update = {{}};
      shapeIndices.forEach((shapeIndex) => {{
        update[`shapes[${{shapeIndex}}].visible`] = visible;
        if (xValue !== undefined && xValue !== null) {{
          update[`shapes[${{shapeIndex}}].x0`] = xValue;
          update[`shapes[${{shapeIndex}}].x1`] = xValue;
        }}
      }});
      Plotly.relayout(graph, update);
    }}

    graph.on('plotly_hover', function(evt) {{
      const point = evt && evt.points && evt.points[0];
      if (!point) {{
        return;
      }}
      setLine(point.x, true);
    }});

    graph.on('plotly_unhover', function() {{
      setLine(null, false);
    }});
  }}

  attach();
}})();
</script>
"""
    display(HTML(html))
def render_shared_hover_figure(fig: go.Figure) -> None:
    wrapper_id = f"burst-hover-{uuid.uuid4().hex}"
    graph_html = fig.to_html(
        full_html=False,
        include_plotlyjs=True,
        config={"responsive": True, "scrollZoom": True, "displaylogo": False},
    )
    html = f"""
<div id="{wrapper_id}" style="margin-bottom: 1rem;">
  {graph_html}
</div>
<script>
(function() {{
  const root = document.getElementById({json.dumps(wrapper_id)});
  if (!root) {{
    return;
  }}

  function attach() {{
    const graph = root.querySelector('.plotly-graph-div');
    if (!graph || typeof graph.on !== 'function') {{
      setTimeout(attach, 50);
      return;
    }}

    const guideCount = 6;
    const shapeStart = Array.isArray(graph.layout.shapes) ? graph.layout.shapes.length - guideCount : -1;
    if (shapeStart < 0) {{
      return;
    }}

    const shapeIndices = Array.from({{ length: guideCount }}, (_, idx) => shapeStart + idx);

    function setLine(xValue, visible) {{
      const update = {{}};
      shapeIndices.forEach((shapeIndex) => {{
        update[`shapes[${{shapeIndex}}].visible`] = visible;
        if (xValue !== undefined && xValue !== null) {{
          update[`shapes[${{shapeIndex}}].x0`] = xValue;
          update[`shapes[${{shapeIndex}}].x1`] = xValue;
        }}
      }});
      Plotly.relayout(graph, update);
    }}

    graph.on('plotly_hover', function(evt) {{
      const point = evt && evt.points && evt.points[0];
      if (!point) {{
        return;
      }}
      setLine(point.x, true);
    }});

    graph.on('plotly_unhover', function() {{
      setLine(null, false);
    }});
  }}

  attach();
}})();
</script>
"""
    display(HTML(html))


## Interactive Burst Plot

Use `N` to choose how many recent trades define the event-time window.

The chart now shows seven panels:

- midprice with burst markers
- signed share over the last `N` trades
- trades per second over the last `N` trades
- trades per second to baseline over the same window, where values above `1.0` mean the current rate is elevated relative to the recent local baseline
- total traded volume over the last `N` trades
- volume imbalance over the last `N` trades, normalized by total volume
- volume to baseline over the same window, where values above `1.0` mean the window is larger than the recent local baseline

Hovering the chart shows a shared dashed cursor guide across all panels so the same event is easy to compare across plots.

The burst markers are intentionally strict by default: the window must be directional, intense, and preceded by a lower-intensity period. Zoom into marker clusters to inspect what price did immediately before and after the burst.


In [6]:
def draw_sudden_burst_plot(
    event_n: int = DEFAULT_EVENT_N,
    imbalance_threshold: float = DEFAULT_IMBALANCE_THRESHOLD,
    intensity_tail_q: float = DEFAULT_INTENSITY_TAIL_Q,
    calm_q: float = DEFAULT_CALM_Q,
    calm_lookback_min: int = DEFAULT_CALM_LOOKBACK_MIN,
    cooldown_sec: int = DEFAULT_BURST_COOLDOWN_SEC,
):
    features = build_event_burst_features(trade_frame, event_n=int(event_n))
    features, thresholds = add_sudden_burst_flags(
        features,
        imbalance_threshold=float(imbalance_threshold),
        intensity_tail_q=float(intensity_tail_q),
        calm_q=float(calm_q),
        calm_lookback_min=int(calm_lookback_min),
        cooldown_sec=int(cooldown_sec),
    )
    summary = pd.Series({
        "event_n": int(event_n),
        "imbalance_threshold": float(imbalance_threshold),
        "intensity_tail_q": float(intensity_tail_q),
        "calm_q": float(calm_q),
        "calm_lookback_min": int(calm_lookback_min),
        "cooldown_sec": int(cooldown_sec),
        "baseline_windows": int(DEFAULT_VOLUME_BASELINE_WINDOWS),
    })
    display(summary.to_frame("value"))
    display(pd.Series(thresholds).to_frame("value"))
    fig = make_sudden_burst_figure(
        features,
        thresholds=thresholds,
        title=(
            f"{SYMBOL} {REFERENCE_DAY}: sudden sign bursts over last {int(event_n)} trades "
            f"(tail q={float(intensity_tail_q):.2f}, baseline={int(DEFAULT_VOLUME_BASELINE_WINDOWS)})"
        ),
    )
    render_shared_hover_figure(fig)
    global latest_burst_features, latest_burst_thresholds, latest_burst_config
    latest_burst_features = features
    latest_burst_thresholds = thresholds
    latest_burst_config = summary
    return features


N_WIDGET = widgets.IntSlider(value=DEFAULT_EVENT_N, min=20, max=500, step=10, description="N trades", continuous_update=False)
IMB_WIDGET = widgets.FloatSlider(value=DEFAULT_IMBALANCE_THRESHOLD, min=0.10, max=1.00, step=0.05, description="imb th", continuous_update=False)
RATE_WIDGET = widgets.FloatSlider(value=DEFAULT_INTENSITY_TAIL_Q, min=0.50, max=0.99, step=0.01, description="rate q", continuous_update=False)
CALM_Q_WIDGET = widgets.FloatSlider(value=DEFAULT_CALM_Q, min=0.05, max=0.90, step=0.05, description="calm q", continuous_update=False)
CALM_MIN_WIDGET = widgets.IntSlider(value=DEFAULT_CALM_LOOKBACK_MIN, min=1, max=60, step=1, description="calm min", continuous_update=False)
COOLDOWN_WIDGET = widgets.IntSlider(value=DEFAULT_BURST_COOLDOWN_SEC, min=0, max=600, step=10, description="cooldown s", continuous_update=False)
REFRESH_BUTTON = widgets.Button(description="Refresh plot", button_style="primary")
REFRESH_OUTPUT = widgets.Output()


def _refresh_burst_plot(_=None):
    with REFRESH_OUTPUT:
        REFRESH_OUTPUT.clear_output(wait=True)
        draw_sudden_burst_plot(
            event_n=N_WIDGET.value,
            imbalance_threshold=IMB_WIDGET.value,
            intensity_tail_q=RATE_WIDGET.value,
            calm_q=CALM_Q_WIDGET.value,
            calm_lookback_min=CALM_MIN_WIDGET.value,
            cooldown_sec=COOLDOWN_WIDGET.value,
        )


REFRESH_BUTTON.on_click(_refresh_burst_plot)
display(widgets.VBox([
    widgets.HBox([N_WIDGET, IMB_WIDGET]),
    widgets.HBox([RATE_WIDGET, CALM_Q_WIDGET]),
    widgets.HBox([CALM_MIN_WIDGET, COOLDOWN_WIDGET]),
    REFRESH_BUTTON,
    REFRESH_OUTPUT,
]))
_refresh_burst_plot()

#Plot controls:

#- `N trades`: last-`N` window used for imbalance, volume, and intensity.
#- `imb th`: minimum absolute signed share required for a directional burst.
#- `rate q`: quantile used to define high activity in trades/sec.
#- `calm q`: quantile used to define the prior calm regime.
#- `calm min`: lookback length for the calm-state median in minutes.
#- `cooldown s`: suppression window after a burst start so the same cluster is not marked repeatedly.

#The thresholds are intentionally simple so the plot is easy to reason about. They are not a trading rule.

## Burst Candidate Table

After using the interactive plot, run this cell to inspect the currently selected burst candidates as rows. The table keeps the same simple definitions as the chart.

In [7]:
FEATURES_FOR_TABLE_N = 100
FEATURES_FOR_TABLE_IMBALANCE_THRESHOLD = 0.60
FEATURES_FOR_TABLE_INTENSITY_TAIL_Q = 0.90
FEATURES_FOR_TABLE_CALM_Q = 0.50
FEATURES_FOR_TABLE_CALM_LOOKBACK_MIN = 5
FEATURES_FOR_TABLE_COOLDOWN_SEC = 60

features_for_table = build_event_burst_features(trade_frame, event_n=FEATURES_FOR_TABLE_N)
features_for_table, table_thresholds = add_sudden_burst_flags(
    features_for_table,
    imbalance_threshold=FEATURES_FOR_TABLE_IMBALANCE_THRESHOLD,
    intensity_tail_q=FEATURES_FOR_TABLE_INTENSITY_TAIL_Q,
    calm_q=FEATURES_FOR_TABLE_CALM_Q,
    calm_lookback_min=FEATURES_FOR_TABLE_CALM_LOOKBACK_MIN,
    cooldown_sec=FEATURES_FOR_TABLE_COOLDOWN_SEC,
)

burst_table = features_for_table.loc[features_for_table["is_sudden_burst_start"], [
    "ts",
    "mid_at_book",
    "signed_share_N",
    "net_count_N",
    "trades_per_sec_N",
    "trades_per_sec_baseline_N",
    "trades_per_sec_to_baseline_N",
    "prior_calm_trades_per_sec_N",
    "event_span_sec_N",
    "volume_imbalance_N",
    "total_volume_N",
    "volume_baseline_N",
    "volume_to_baseline_N",
    "burst_side",
]].copy()

burst_table["rank_abs_signed_share"] = burst_table["signed_share_N"].abs().rank(ascending=False, method="first")

display(pd.Series(table_thresholds).to_frame("value"))
display(burst_table.sort_values(["ts"]).head(50))
display(burst_table.sort_values(["trades_per_sec_N"], ascending=False).head(50))

,value
high_intensity_cutoff,171.821306
calm_intensity_cutoff,18.125793
imbalance_threshold,0.600000
cooldown_sec,60.000000
raw_sudden_burst_windows,24560.000000
sudden_burst_start_count,240.000000
buy_burst_start_count,126.000000
sell_burst_start_count,114.000000


,ts,mid_at_book,signed_share_N,net_count_N,trades_per_sec_N,trades_per_sec_baseline_N,trades_per_sec_to_baseline_N,prior_calm_trades_per_sec_N,event_span_sec_N,volume_imbalance_N,total_volume_N,volume_baseline_N,volume_to_baseline_N,burst_side,rank_abs_signed_share
68188,2026-02-23 01:32:07.870000+00:00,65185.145,0.98,98.0,248.138958,57.937428,4.282878,15.599407,0.403,0.999781,5.11221,5.033995,1.015537,buy,155.0
72305,2026-02-23 01:37:48.036000+00:00,65282.810,0.96,96.0,176.056338,84.033613,2.095070,12.118274,0.568,0.915452,0.72503,0.780340,0.929121,buy,185.0
74350,2026-02-23 01:40:54.809000+00:00,65202.535,-0.74,-74.0,202.839757,43.811612,4.629817,11.228385,0.493,-0.933675,3.60646,3.577550,1.008081,sell,228.0
76222,2026-02-23 01:45:08.082000+00:00,65132.325,-1.00,-100.0,719.424460,57.359941,12.542280,7.574610,0.139,-1.000000,2.30599,2.315965,0.995693,sell,1.0
77844,2026-02-23 01:46:36.232000+00:00,64960.015,-0.66,-66.0,1265.822785,1183.473389,1.069583,11.541364,0.079,-0.746646,1.16793,1.245680,0.937584,sell,235.0
110738,2026-02-23 02:10:31.622000+00:00,64726.870,-0.94,-94.0,178.253119,41.381620,4.307543,7.940289,0.561,-0.543604,0.44549,0.451885,0.985848,sell,195.0
113518,2026-02-23 02:16:08.921000+00:00,64743.520,1.00,100.0,213.219616,22.581010,9.442431,8.996851,0.469,1.000000,1.53133,1.203580,1.272313,buy,2.0
116833,2026-02-23 02:24:43.505000+00:00,64542.015,-0.70,-70.0,7142.857143,46.008744,155.249992,6.873088,0.014,-0.239168,10.94157,9.977625,1.096611,sell,233.0
118584,2026-02-23 02:30:03.519000+00:00,64765.945,0.86,86.0,257.731959,140.449438,1.835052,5.413471,0.388,0.956382,11.40673,8.773470,1.300139,buy,216.0
119399,2026-02-23 02:31:55.943000+00:00,64857.135,1.00,100.0,201.207243,7.792232,25.821515,6.784721,0.497,1.000000,0.90107,0.885940,1.017078,buy,3.0


,ts,mid_at_book,signed_share_N,net_count_N,trades_per_sec_N,trades_per_sec_baseline_N,trades_per_sec_to_baseline_N,prior_calm_trades_per_sec_N,event_span_sec_N,volume_imbalance_N,total_volume_N,volume_baseline_N,volume_to_baseline_N,burst_side,rank_abs_signed_share
116833,2026-02-23 02:24:43.505000+00:00,64542.015,-0.70,-70.0,7142.857143,46.008744,155.249992,6.873088,0.014,-0.239168,10.94157,9.977625,1.096611,sell,233.0
398248,2026-02-23 14:23:02.545000+00:00,66091.745,-1.00,-100.0,5555.555556,32.226877,172.388889,7.916403,0.018,-1.000000,0.02324,0.015490,1.500323,sell,63.0
263110,2026-02-23 10:38:35.774000+00:00,66250.005,-0.98,-98.0,5000.000000,42.283298,118.250000,2.384985,0.020,-0.996701,3.43172,2.791390,1.229395,sell,160.0
180683,2026-02-23 06:11:17.092000+00:00,65221.995,1.00,100.0,3703.703704,99.700897,37.148148,6.957974,0.027,1.000000,1.30994,1.513775,0.865347,buy,11.0
379340,2026-02-23 13:51:12.675000+00:00,66372.005,-1.00,-100.0,3448.275862,41.511000,83.068966,11.442957,0.029,-1.000000,0.14577,0.170050,0.857218,sell,53.0
826966,2026-02-23 22:05:22.851000+00:00,64758.225,-1.00,-100.0,3225.806452,23.975066,134.548387,14.643432,0.031,-1.000000,0.23565,0.200215,1.176985,sell,134.0
742355,2026-02-23 20:13:36.472000+00:00,64054.855,-1.00,-100.0,3030.303030,127.877238,23.696970,18.057060,0.033,-1.000000,1.60792,1.719115,0.935318,sell,103.0
277839,2026-02-23 11:46:37.549000+00:00,66170.785,-1.00,-100.0,2702.702703,56.666028,47.695291,15.268341,0.037,-1.000000,0.00800,0.713520,0.011212,sell,27.0
392296,2026-02-23 14:11:45.191000+00:00,66267.385,-1.00,-100.0,2631.578947,35.727045,73.657895,7.619048,0.038,-1.000000,0.09918,0.103020,0.962726,sell,59.0
858318,2026-02-23 23:02:03.291000+00:00,64710.005,1.00,100.0,2500.000000,6.806251,367.309379,16.398819,0.040,1.000000,0.01614,0.037430,0.431205,buy,149.0
